# Python for Neuroscience (2026)

# 03. Patch-clamp: explore datasets

Python is a powerful tool for exploring data stored in **DataFrames** using the [Pandas](https://pandas.pydata.org/) library.

**GOAL**: Get the basics to navigate tables with Pandas, and plot data using [Matplotlib](https://matplotlib.org/) and [Seaborn](https://seaborn.pydata.org/), and perform statistical tests using [SciPy](https://docs.scipy.org/doc/scipy/reference/stats.html).


**Further reading and resources**
* [Common statistical tests are linear models](https://lindeloev.github.io/tests-as-linear/) by Jonas Kristoffer Lindeløv.
* [Learning statistics with Python](https://ethanweed.github.io/pythonbook/landingpage.html) by Danielle Navarro and Ethan Wood.
* [Neural Data Science in Python](https://neuraldatascience.io/intro.html) by Aaron J Newman.
* [Pandas Cookbook](https://github.com/jvns/pandas-cookbook) by Jake VanderPlas.
* [Pandas Notebooks](https://github.com/plembo/pandas-tutorials) by Corey Schafer's.
* [The Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/) by Jake VanderPlas. See [Visualization with Matplotlib](https://jakevdp.github.io/PythonDataScienceHandbook/04.00-introduction-to-matplotlib.html), [Visualization with Seaborn](https://jakevdp.github.io/PythonDataScienceHandbook/04.14-visualization-with-seaborn.html), and [Data Manipulation with Pandas](https://jakevdp.github.io/PythonDataScienceHandbook/03.00-introduction-to-pandas.html).


# Example data

In this notebook, I will use the brain cell database from the **Allen Institute for Brain Science** as an example dataset.  You can also download the table directly from the [Allen Brain Atlas website](https://celltypes.brain-map.org/data) by clicking **"Download Cell Feature Data."** This dataset contains electrophysiological and morphological data obtained from patch-clamp recordings in both mouse and human brains. These features were used to classify neurons into distinct subtypes (**Figure 1**), which is one of the main challenges in neuroscience because of the brain’s high cellular diversity.  

Defining a cell type is not trivial, and morphoelectrical features can be combined with transcriptomic data for a more integrative classification ([Gouwens et al., 2020](https://pubmed.ncbi.nlm.nih.gov/33186530/), [Scala et al., 2021](https://www.nature.com/articles/s41586-020-2907-3)). The goal is to identify cell types with similar attributes and functions. For example, although **inhibitory interneurons** ([Tremblay et al., 2016](https://www.sciencedirect.com/science/article/pii/S0896627316303117)) are fewer than excitatory neurons in the cortex (about 20% vs. 80%), there are at least four major classes of inhibitory neurons with distinct intrinsic and functional properties: Pvalb, Vip, Sst, and Lamp5 (**Figure 1b**).

<img src="Images/03_Neuronal-diversity_Gouwens_2019.png" width="800">

**Figure 1.** Examples of excitatory (a) and inhibitory (b) neurons in the mouse visual cortex. Top panels: morphological reconstructions of dendrites.  Bottom panels: electrophysiological responses from the same neurons to hyperpolarizing and depolarizing current injection.  Source: [Gouwens et al., 2019](https://pmc.ncbi.nlm.nih.gov/articles/PMC8078853/).


You can explore the Allen Institute Cell Database using the [interactive website](https://celltypes.brain-map.org/data) or the [Allen Software Development Kit (SDK)](https://allensdk.readthedocs.io/en/latest/).  Through the Allen SDK, you can access all available electrophysiology measurements and morphological reconstructions (see links below).

**Additional resources and reading**
* [Cell Types Database](https://alleninstitute.github.io/AllenSDK/_static/examples/nb/cell_types.html). Example notebooks and documentation.  
* [Open Neuroscience Education](https://sites.google.com/ucsd.edu/neuroedu/cell-types/jupyter-instructions). Excellent teaching resources created by Dr. Ashley Juavinett.  
* [Transgenic mouse lines](https://portal.brain-map.org/explore/toolkit/mice). The Allen Institute for Brain Science used [Cre mouse lines](https://spikesandbursts.wordpress.com/2018/07/18/cre-mice-for-targeting-neurons/) to identify genetically defined cell classes. Driver lines are used to target specific populations, while reporter lines are used to visualize (with fluorescent proteins) or manipulate those populations.
* [Zheng et al., 2022](https://www.sciencedirect.com/science/article/pii/S0092867422007838). What is a cell type and how to define it?

# Import the packages

In [ ]:
import os

# Dataframes
import pandas as pd
import numpy as np

# Optional: Display all columns in table
pd.set_option('display.max_columns', None)

# Display all or a specific number of rows
# pd.set_option('display.max_rows', 20)

# Stats libraries
import scipy.stats as stats
from scipy.stats import shapiro, bartlett

# Plotting libraries
import seaborn as sns
import matplotlib.pyplot as plt

# Interactive plots (comment out if needed)
# import ipywidgets as widgets
# from IPython.display import display
# #For Jupyter Lab:
# %matplotlib widget

# Paths

**Data management: Resources**
- [FAIR Guiding Principles for scientific data management and stewardship](https://www.go-fair.org/fair-principles/)
- [NIN guidelines for data storage](https://nin.nl/wp-content/uploads/sites/2/2024/10/data-storage-protocol-NIN_20211210.pdf).
- [NIH data management](https://grants.nih.gov/policy-and-compliance/policy-topics/sharing-policies/dms/data-management).
- [Cambridge research data management](https://www.data.cam.ac.uk/about).

In [ ]:
# Change the paths and folder names according to your data structure
notebook_name = 'patch-clamp'

# Data path to 'Data_example' folders.
data_path = os.getcwd()

# Change the folder names accordingly
paths = {'data':  f'{data_path}/Example Data',
         'processed_data': f'{data_path}/Processed_data/{notebook_name}',
         'analysis': f'{data_path}/Analysis/{notebook_name}'}

# Make folders if they do not exist yet
for path in paths.values():
    os.makedirs(path, exist_ok=True)

# Download the data

In [ ]:
if 'google.colab' in str(get_ipython()):

    !wget -O "Example Data/pfc_pvalb_aps_01.abf" \
    "https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Example%20data/cell_types_specimen_details.csv"

else:
    print("Download data from 'Example Data' folder.")

# Plot settings

You can set some general plot settings at the beginning of your notebook.

- Matplotlib: [rcParams](https://matplotlib.org/stable/users/explain/customizing.html#customizing-with-dynamic-rc-settings).
- Seaborn: [set](https://seaborn.pydata.org/generated/seaborn.set.html).

In [ ]:
# Matplotlib settings
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 18,  # Base size used by most elements
    'axes.labelsize': 18,     # Axis labels
    'axes.titlesize': 18,     # Plot titles
    'xtick.labelsize': 18,    # X axis numbers
    'ytick.labelsize': 18,    # Y axis numbers
    'legend.fontsize': 18,    # Legend text
    'savefig.transparent': True,
    'svg.fonttype': 'none',   # Editable text in SVGs
})

# Seaborn settings
sns.set(style="ticks",
        context="notebook",
        font="Arial",
        rc={
            "axes.labelsize": 18,
            "axes.titlesize": 18,
            "xtick.labelsize": 18,
            "ytick.labelsize": 18,
            "legend.fontsize": 18,
            "svg.fonttype": 'none',   # Editable text in SVGs
        })

# Load the dataframe

* For excel files you can use [`pandas.read_excel`](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html).
* For CSV files, you can use [`pandas.read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)


In [ ]:
dataset = pd.read_csv(os.path.join(paths['data'],
                                   'cell_types_specimen_details.csv'))

dataset.head(10)  # Use head or tail to show the first n rows

# Transform the table

Once you have merged the dataset, you may still need to transform it to add calculations or new labels. Common transformations include:

- Transpose the dataframe: `dataframe.T`
- Insert a column: `dataframe.insert` or `dataframe.assign`, insert adds a new column at a specific position, while `assign` creates a new column without modifying the original dataframe unless you assign it to a variable
- Rename a column: `dataframe.rename`

It is recommended to work on a copy of the dataframe and, when saving, use a different name to avoid overwriting the original dataset.

## Insert a column

In [ ]:
# Insert a new column
dataset_processed = dataset.copy()  # Make a copy of the original dataset
dataset_processed.insert(0, 'id', range(1, len(dataset_processed) + 1))
dataset_processed.head(10)

In [ ]:
dataset.assign(id=range(1, len(dataset) + 1))
dataset.head(10)

In [ ]:
# The original dataset has not been changed
dataset.head()

## Q: Rename column 'structure__name' to 'brain_area'

<details>
<summary><strong>Solution</strong></summary>

```python

# Rename a column
dataset_processed = dataset.copy()  # Make a copy of the original dataset
dataset_processed = dataset.rename(
    columns={"structure__name": "brain_area"})

dataset_processed.head(10)
```

# Explore the dataset

Some Pandas functions are useful for getting an overview of large datasets. You can also type `?` after each function in Python for a longer explanation.

* `dataset.shape`. Shows the number of rows and columns (also displayed at the bottom of the table).
* `dataset.columns`. Lists the names of all columns.
* `dataset.info()`. Displays all columns, the number of non-null values, and the data type of each column.
* `dataset.head()`. Returns the first few rows (5 by default).
* `dataset.tail()`. Returns the last few rows (5 by default).
* `dataset.loc[rows, [columns]]`. Select specific rows and columns by label.
* `dataset.sort_values(by='column_name')`. Sorts the dataframe by a specific column (ascending by default).

In [ ]:
dataset.info()

# Filter the table

Filtering the table is not strictly necessary, since you can select specific rows and columns for analysis. However, it can be convenient for many things. For example, showing only mouse data. You can then use the functions above to check the sample size of the filtered subset or combine multiple conditions to select the group of interest (see examples in plots and statistics).

An important note for beginners to avoid the common Python `KeyError`:
* `['column_name']` returns a Pandas Series and can only select one column.
* `[['column_name']]` returns a Pandas DataFrame, which can include one or more columns, but to filter rows using conditions (boolean masks), you need to apply the mask to the DataFrame itself, not to the double-bracket selection.

In [ ]:
dataset_mice = dataset[(dataset['donor__species'] == 'Mus musculus')]
dataset_mice

## Q: Select all the ephys columns

**Tip**: Columns that start with `ef`

<details>
<summary><strong>Solution</strong></summary>

```python

# Option A
# dataset_mice_ephys = dataset_mice[[column for column in dataset_mice.columns if column.startswith('ef')]]

# Option B
dataset_ephys = dataset.loc[:, dataset.columns.str.startswith('ef')]

dataset_ephys.head()
```

## Q: Select all the ephys columns and first three columns

<details>
<summary><strong>Solution</strong></summary>

```python

# Select some columns you want to keep based on position or name
filtered_columns = list(dataset.columns[:3]) + list(dataset.columns[dataset.columns.str.startswith('ef')])

# Create the new DataFrame
dataset_ephys = dataset[filtered_columns]
dataset_ephys.head()
```

## Q: Five highest values of ef__ri in 'line_name' == 'Pvalb-IRES-Cre'

**Tips**: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html

**Q2**: How can you just show the columns of interest?

<details>
<summary><strong>Solution</strong></summary>

```python

dataset[dataset['line_name'] == 'Pvalb-IRES-Cre'].sort_values(
    'ef__ri', ascending=False).head(5)

# R2: Show only the results from those two columns
dataset[dataset['line_name'] == 'Pvalb-IRES-Cre'].sort_values(
    'ef__ri', ascending=False).head(5)[['line_name', 'ef__ri']]

```

# Aggregate the table

Aggregation in Pandas allows you to group data by one or more columns and calculate summary statistics for each group, such as count, mean, or sum. This is often done using `.groupby()` and is useful for summarizing data by categories.

In [ ]:
# Group by structure and feature
groupby_parameter = 'structure__acronym'
ephys_feature = 'ef__avg_firing_rate'

grouped_dataset = (
    dataset.groupby(groupby_parameter)
    .agg(num_cells = (ephys_feature, 'size'),
         av_firing_rate = (ephys_feature, 'mean'))
    .dropna())

grouped_dataset

## Q: av_firing_rate from neurons in V1 (VISp)

**Tip**: Use `startswith('VISp')`

In [ ]:
# Group by structure and feature
groupby_parameter = 'structure__acronym'
ephys_feature = 'ef__avg_firing_rate'

# COMPLETE THE CODE


<details>
<summary><strong>Solution</strong></summary>

```python

# Group by structure and feature
groupby_parameter = 'structure__acronym'
ephys_feature = 'ef__avg_firing_rate'

dataset_area = dataset[dataset['structure__acronym'].str.startswith('VISp')]

grouped_dataset = (
    dataset_area.groupby(groupby_parameter)
    .agg(num_cells = (ephys_feature, 'size'),
         av_firing_rate = (ephys_feature, 'mean'))
    .dropna())

grouped_dataset
```

# Summary statistics

The function `describe` give you a quick overview of common descriptive statistics. You can use it on the entire dataframe or for specific features.

```python
dataset.mean(numeric_only=True)             
dataset.mean(numeric_only=True).to_frame() # Converts the result into a DataFrame

In [ ]:
# Statistics for neuron subtype

neuron_subtype = dataset[
    (dataset.line_name == 'Pvalb-IRES-Cre') &
    (dataset.structure__acronym == 'VISp5')
]

neuron_subtype.describe()

# Plots: Pandas

[Pandas](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html) has built-in plotting functions that are easy to use. They are not as versatile as Matplotlib or Seaborn, but are useful for quickly plot the data.

**Documentation**: [`pandas.plot`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html).

In [ ]:
ephys_feature = 'ef__tau'

feature_means = dataset.groupby('line_name')[ephys_feature].mean()
feature_means.plot(kind='bar', figsize=(12,4))
plt.show()

# Plots: Matplotlib

[Matplotlib](https://matplotlib.org/) and [Seaborn](https://seaborn.pydata.org/) are two powerful Python libraries for data visualization that let you create high-quality figures.  

While Python may not be as specialized for statistics as R, libraries like [SciPy](https://docs.scipy.org/doc/scipy/reference/stats.html) and [statsmodels](https://www.statsmodels.org/stable/index.html) do the job. To compare three or more groups, you can use **one-way ANOVA**. The ANOVA test assumes numeric data, independent samples, normally distributed groups, and similar variances across groups (homogeneity of variance).  

If sample sizes are large enough (usually >25–30), the distribution of the sample means will approximate a Gaussian distribution even if the population is not perfectly normal (Central Limit Theorem). If the assumptions of ANOVA are violated, you can use the non-parametric alternative, the [Kruskal-Wallis test](https://www.graphpad.com/guides/prism/latest/statistics/stat_checklist_kw.htm), which does not require normally distributed data.


## Plot 3 groups

In [ ]:
# Load dataset
dataset = pd.read_csv(os.path.join(paths['data'], 'cell_types_specimen_details.csv'))

# Filter dataset for your region of interest
filtered_dataset = dataset[
    (dataset['donor__species'] == 'Mus musculus') &
    (dataset['structure__acronym'] == 'VISp2/3')
]

# Define the groups to compare
group_labels = ['Pvalb-IRES-Cre', 'Sst-IRES-Cre', 'Vip-IRES-Cre']

# Feature for comparison
feature = 'ef__avg_firing_rate'

# Extract feature values for each group (drop NaNs)
group_data = [
    filtered_dataset[filtered_dataset['line_name'] == label][feature].dropna()
    for label in group_labels
]

# Boxplot
fig, ax = plt.subplots(figsize=(4,4))
ax.boxplot(group_data, labels=[lbl.split('-')[0] for lbl in group_labels])
ax.set_ylabel("Average firing rate (Hz)")
plt.show()

# Save figure
fig.tight_layout()
plot_path = f"{paths['analysis']}VIS_interneurons_{feature}.png"
fig.savefig(plot_path, dpi=300)

# ANOVA
anova_results = stats.f_oneway(*group_data)
p_str = f"{anova_results.pvalue:.3f}" if anova_results.pvalue >= 0.0001 else "<0.0001"
print(f"\nOne-way ANOVA: F = {anova_results.statistic:.3f}, p = {p_str}")

## Q: Create a table with summary statistics

**Tips**:
- Calculate n, mean, std.
- Optional: Add Shapiro-Wilk test for test normal distribution: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.shapiro.html

In [ ]:
# CREATED DATAFRAME
group_stats = pd.DataFrame(columns=[# COMPLETE])

# Loop over groups
for i, group in enumerate(group_data):
    group_stats.loc[group_labels[i], 'n'] = len(group)
    # COMPLETE THE CODE
    group_stats.loc[group_labels[i], 'mean'] = group.mean()
    group_stats.loc[group_labels[i], 'std'] = group.std()

    # ADD SHAPIRO-WILK TEST
    W, p = # COMPLETE
    # COMPLETE CODE

# ANOVA RESULTS
anova_results = stats.f_oneway(*group_data)
p_str = f"{anova_results.pvalue:.3f}" if anova_results.pvalue >= 0.0001 else "<0.0001"
print(f"\nOne-way ANOVA: F = {anova_results.statistic:.3f}, p = {p_str}")

group_stats

<details>
<summary><strong>Solution</strong></summary>

```python

# Statistics
group_stats = pd.DataFrame(columns=['n', 'mean', 'std', 'Shapiro_W', 'Shapiro_p'])  # set index here

# Loop over groups
for i, group in enumerate(group_data):
    group_stats.loc[group_labels[i], 'n'] = len(group)
    group_stats.loc[group_labels[i], 'mean'] = group.mean()
    group_stats.loc[group_labels[i], 'std'] = group.std()

    # Shapiro-Wilk test
    W, p = shapiro(group)
    group_stats.loc[group_labels[i], 'Shapiro_W'] = W
    group_stats.loc[group_labels[i], 'Shapiro_p'] = p

# ANOVA
anova_results = stats.f_oneway(*group_data)
p_str = f"{anova_results.pvalue:.3f}" if anova_results.pvalue >= 0.0001 else "<0.0001"
print(f"\nOne-way ANOVA: F = {anova_results.statistic:.3f}, p = {p_str}")

group_stats
```

# Plots: Seaborn

[Seaborn](https://seaborn.pydata.org/) is built on Matplotlib and offers more flexibility and higher-level plotting functions to create more complex plots with fewer lines. One useful feature is the [pairplot](https://seaborn.pydata.org/generated/seaborn.pairplot.html) feature to plot pairwise relationships in a dataset.


## Pairwise correlations

In [ ]:
filtered_dataset = dataset[
    (dataset['donor__species'] == 'Mus musculus') &
    (dataset['structure__acronym'].str.startswith('VISp')) &
    (dataset['line_name'] == 'Vip-IRES-Cre')]

columns_pairplot = ['ef__f_i_curve_slope',
                   'ef__avg_firing_rate',
                   'ef__tau',
                   'ef__ri']

# Create the pair plot
pair_plot = sns.pairplot(filtered_dataset[columns_pairplot], diag_kind='hist')

# Adjust layout and show
pair_plot.fig.tight_layout()
plt.show()

# Save figure
plot_path = f"{paths['analysis']}VISp_pairplots.png"  # or .svg
pair_plot.fig.savefig(plot_path, dpi=300)

## Q: Boxplot of 'ef__ri' in mouse 'VISp2/3'

**Tips**:
- Boxplot: https://seaborn.pydata.org/generated/seaborn.boxplot.html
- dataset['donor__species'] == 'Mus musculus'
- dataset['structure__acronym'] == 'VISp2/3'
- feature = 'ef__ri'

## Plot

In [ ]:
# Load
dataset = pd.read_csv(os.path.join(paths['data'], 'cell_types_specimen_details.csv'))

filtered_dataset = (dataset[
    (# INSERT DONOR SPECIES) &
    (# INSERT STRUCTURE ACRONYM)])  # Columns to display between [[]]

x_categories = 'line_name'
y_feature = # ADD FEATURE

# Create figure and axes
fig, ax = plt.subplots(figsize=(16, 10))

# ADD SEABORN BOXPLOT
# COMPLETE THE CODE

# Rotate x-axis labels
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Set axis labels
ax.set_xlabel(x_categories)
ax.set_ylabel(y_feature)

# Adjust layout
fig.tight_layout()

# Save figure
plot_path = f"{paths['analysis']}VISp_L23_{x_categories}_{y_feature}.png"  # or .svg
fig.savefig(plot_path, dpi=300)

# Show the figure
plt.show()


<details>
<summary><strong>Solution</strong></summary>

```python

# Load
dataset = pd.read_csv(os.path.join(paths['data'], 'cell_types_specimen_details.csv'))

filtered_dataset = (dataset[
    (dataset['donor__species'] == 'Mus musculus') &
    (dataset['structure__acronym'] == 'VISp2/3')])  # Columns to display between [[]]

x_categories = 'line_name'
y_feature = 'ef__ri'

# Create figure and axes
fig, ax = plt.subplots(figsize=(16, 10))

# Plot the boxplot on the Axes
sns.boxplot(x=x_categories, y=y_feature, data=filtered_dataset, ax=ax)

# Rotate x-axis labels
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Set axis labels
ax.set_xlabel(x_categories)
ax.set_ylabel(y_feature)

# Adjust layout
fig.tight_layout()

# Save figure
plot_path = f"{paths['analysis']}VISp_L23_{x_categories}_{y_feature}.png"  # or .svg
fig.savefig(plot_path, dpi=300)

# Show the figure
plt.show()

```